# OPE bootstrap and Jacobi identities

Two tools for universal vertex algebras (strong generators + declared OPEs, given to the constructor `UniversalVertexAlgebra(R, generators, opes, ...)`):

* `V.jacobi_identity_failures()` / `V.satisfies_jacobi_identity()` / `UniversalVertexAlgebra(..., verify_jacobi=True)`: do the declared OPEs define a vertex algebra (with the ordered normally ordered monomials as a basis)?
* `OPEBootstrap(V)` (`bootstrap.py`): give the generators conformal weights (`degrees=`), declare the OPEs you know (e.g. Virasoro and primary fields), and every undeclared OPE gets the most general ansatz allowed by the weights (and parity, and a grading if set), with unknown coefficients `x0, x1, ...`. The Jacobi identities become polynomial equations in the unknowns; `impose_jacobi()` imposes them triple of generators by triple (eliminating unknowns as it goes), `solve()` gives the irreducible components of the solutions and `S.algebra()` the vertex algebra of a component.

Coefficients: over a field (`QQ`, `Frac(QQ['c'])`) the central charge is generic; over a polynomial ring (`QQ['c']`) it is a parameter that is solved for, so special values show up as separate components.

**Prerequisites:** `01_getting_started.ipynb`.

**Contents**

1. Checking declared OPEs: the $W_3$ algebra
2. Bootstrapping $W_3$
3. The central charge as a parameter
4. Square roots: $W(2,4)$
5. The $N=2$ superconformal algebra (odd generators, a grading)
6. $W(2,3,4)$
7. Saving and resuming a bootstrap

In [1]:
import vertex_algebras as va
from vertex_algebras import *
import os, sys, time
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
from vertex_algebras.bootstrap import OPEBootstrap

## 1. Checking declared OPEs: the $W_3$ algebra

$W(z)W(w) \sim \frac{c/3}{(z-w)^6} + \frac{2L}{(z-w)^4} + \frac{\partial L}{(z-w)^3} + \frac{2\beta\Lambda + \frac{3}{10}\partial^2 L}{(z-w)^2} + \frac{\beta\partial\Lambda + \frac{1}{15}\partial^3 L}{z-w}$ with $\Lambda = {:}LL{:} - \frac{3}{10}\partial^2 L$ is consistent exactly for $\beta = 16/(22+5c)$. The coefficients are written as normally ordered expressions in $L$, $W$ (words of fields, right-nested), before the algebra exists; the constructor normally orders them with the OPEs themselves ($\partial\Lambda$ contains ${:}L\,\partial L{:}$, whose basis form needs the $L$–$L$ OPE).

In [2]:
K.<c> = Frac(QQ['c'])
E.<L, W> = NormallyOrderedExpressions(K)
VIR = {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W]}
Lam = L*L - 3/10*L.derivative(2)

def w3(beta):
    WW = [L.derivative(3)/15 + beta*Lam.derivative(), 3/10*L.derivative(2) + 2*beta*Lam,
          L.derivative(), 2*L, 0, c/3]
    return dict(VIR) | {('W', 'W'): WW}

V = UniversalVertexAlgebra(K, E, w3(16/(22 + 5*c)))
print('beta = 16/(22+5c):', V.satisfies_jacobi_identity())
V = UniversalVertexAlgebra(K, E, w3(1))
bad = V.jacobi_identity_failures()
print('beta = 1:', len(bad), 'failing identities, e.g.')
for label in list(bad)[:3]:
    print('   ', label, ':', bad[label])

beta = 16/(22+5c): True
beta = 1: 18 failing identities, e.g.
    ('jacobi', 'L', 'W', 'W', 3, 0) : (c + 6/5)*dL
    ('jacobi', 'L', 'W', 'W', 3, 1) : (2*c + 12/5)*L
    ('jacobi', 'L', 'W', 'W', 4, 0) : (4*c + 24/5)*L


Labels: `('jacobi', a, b, c, m, n)` is the commutator formula $a_{(m)}(b_{(n)}c) - (-1)^{|a||b|} b_{(n)}(a_{(m)}c) - \sum_j \binom{m}{j}(a_{(j)}b)_{(m+n-j)}c$; `('skew', a, b, n)` is skew-symmetry of $a_{(n)}b$ for OPEs declared directly (a generator with itself, or both orientations). With `verify_jacobi=True` the constructor refuses inconsistent OPEs:

In [3]:
try:
    UniversalVertexAlgebra(K, E, w3(1), verify_jacobi=True)
except ValueError as e:
    print(str(e)[:200])
print(UniversalVertexAlgebra(K, E, w3(16/(22 + 5*c)), verify_jacobi=True).satisfies_jacobi_identity())

the OPEs violate the Jacobi identity [L_(3), W_(0)] W: (c + 6/5)*dL (and at least 2 more)
True


## 2. Bootstrapping $W_3$

Only $L$–$L$ and $L$–$W$ are declared. The $W$–$W$ ansatz has 11 unknowns; skew-symmetry removes 6 at once, Virasoro covariance (the triples with $L$) all but the normalisation `x0`, which we fix to $c/3$.

In [4]:
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W]},
                           degrees={'L': 2, 'W': 3})
B = OPEBootstrap(V)
print(B)
print(B.ansatz())
print([(x, B.describe(x)) for x in B.unknowns()])

OPE bootstrap of Universal vertex algebra on generators ['L', 'W'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field: 11 unknowns in the OPEs of [('W', 'W')] (5 left), 0 equations, Jacobi identities imposed for 0 of 8 triples
{('W', 'W'): [x4*:dL L: + (-1/24*x1 + 1/12*x4 + 1/2*x5)*d^3L + 1/2*x6*d^2W, x4*:L L: + x5*d^2L + x6*dW, 1/2*x1*dL, x1*L, 0, x0]}
[(x0, 'coefficient of 1 in W_(5) W'), (x1, 'coefficient of L in W_(3) W'), (x4, 'coefficient of :L L: in W_(1) W'), (x5, 'coefficient of d^2L in W_(1) W'), (x6, 'coefficient of dW in W_(1) W')]


In [5]:
B.fix({'x0': c/3})
B.impose_jacobi(verbose=True)
[S] = B.solve()
A = S.algebra()
for k, p in enumerate(A.opes()[('W', 'W')]):
    print('W_(%d) W =' % k, p)
print(A.satisfies_jacobi_identity())

Jacobi (L, L, L): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (L, L, W): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (L, W, L): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (W, L, L): 0 failing identities, 4 unknowns left, 0 equations, 0.0 s


Jacobi (L, W, W): 9 failing identities, 0 unknowns left, 0 equations, 0.0 s


Jacobi (W, L, W): 0 failing identities, 0 unknowns left, 0 equations, 0.0 s


Jacobi (W, W, L): 0 failing identities, 0 unknowns left, 0 equations, 0.0 s


Jacobi (W, W, W): 0 failing identities, 0 unknowns left, 0 equations, 0.0 s


W_(0) W = (32/5/(c + 22/5))*:dL L: + ((1/15*c - 2/15)/(c + 22/5))*d^3L
W_(1) W = (32/5/(c + 22/5))*:L L: + ((3/10*c - 3/5)/(c + 22/5))*d^2L
W_(2) W = dL
W_(3) W = 2*L
W_(4) W = 0
W_(5) W = 1/3*c
True


## 3. The central charge as a parameter

Over `QQ['c']`, $c$ is solved for like the unknowns. Without a normalisation the $W_3$ algebras form one family, polynomial in $c$ and the `:L L:` coefficient `x4`; the two-point function `x0` vanishes at $c = -22/5$.

In [6]:
P.<c> = QQ['c']
E.<L, W> = NormallyOrderedExpressions(P)
V = UniversalVertexAlgebra(P, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W]},
                           degrees={'L': 2, 'W': 3})
B = OPEBootstrap(V)
B.impose_jacobi()
print(B.solve())
print('x0 =', B.eliminated()[B.ring()('x0')].factor())

[Bootstrap solution 1 of 1: free [c, x4]]
x0 = (1/96) * x4 * c * (5*c + 22)


A primary of weight 5 is consistent (exactly, without null fields) only for $c^2 - 268c - 44 = 0$, i.e. $c = 134 \pm 60\sqrt5$; the solution algebra lives over that number field.

In [7]:
E.<L, X> = NormallyOrderedExpressions(P)
V = UniversalVertexAlgebra(P, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'X'): [X.derivative(), 5*X]},
                           degrees={'L': 2, 'X': 5})
B = OPEBootstrap(V)
B.fix({'x0': c/5})
t = time.time(); B.impose_jacobi(); print('%.1f s' % (time.time() - t))
print(B.equations())
for S in B.solve():
    print(S)
S = B.solve()[0]
A = S.algebra()
print(A.base_ring(), A.satisfies_jacobi_identity())

0.2 s
[c^3 - 268*c^2 - 44*c]
Bootstrap solution 1 of 2: no free unknowns, c^2 - 268*c - 44 = 0
Bootstrap solution 2 of 2: no free unknowns, c = 0


Number Field in c with defining polynomial c^2 - 268*c - 44 True


## 4. Square roots: $W(2,4)$

With $W_{(7)}W = c/4$ the self-coupling of the weight-4 primary is $\sqrt{54(c+24)(c^2-172c+196)/((5c+22)(7c+68)(2c-1))}$: the component is rational only over an extension of $\mathbb{Q}(c)$, and `algebra()` works over it.

In [8]:
K.<c> = Frac(QQ['c'])
E.<L, W> = NormallyOrderedExpressions(K)
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 4*W]},
                           degrees={'L': 2, 'W': 4})
B = OPEBootstrap(V)
B.fix({'x0': c/4})
B.impose_jacobi()
[S] = B.solve()
print(S.algebraic_variable(), '=', B.describe(S.algebraic_variable()))
p = S.extension().modulus()
print((-p(0)).factor())
print(S.algebra().satisfies_jacobi_identity())

x5 = coefficient of W in W_(3) W
(27/35) * (c - 1/2)^-1 * (c + 22/5)^-1 * (c + 68/7)^-1 * (c + 24) * (c^2 - 172*c + 196)


True


## 5. The $N=2$ superconformal algebra (odd generators, a grading)

$L$, a weight-1 current $J$ and odd primaries $G^\pm$ of weight $3/2$; the $U(1)$ charge grading keeps the ansatz small. Three components: the superconformal algebra (with free normalisations `x2`, `x5`) and two degenerate ones.

In [9]:
E.<L, J, Gp, Gm> = NormallyOrderedExpressions(K, parities={'Gp': 1, 'Gm': 1})
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'J'): [J.derivative(), J],
                                  ('L', 'Gp'): [Gp.derivative(), 3/2*Gp], ('L', 'Gm'): [Gm.derivative(), 3/2*Gm]},
                           degrees={'L': 2, 'J': 1, 'Gp': 3/2, 'Gm': 3/2},
                           horizontal_grading={'Gp': 1, 'Gm': -1})
B = OPEBootstrap(V)
B.impose_jacobi()
print(B.ansatz())
for S in B.solve():
    print(S)

{('J', 'J'): [0, x0], ('J', 'Gp'): [x2*Gp], ('J', 'Gm'): [x3*Gm], ('Gp', 'Gp'): [], ('Gp', 'Gm'): [(x2*x5 - 2*x0*x7)*L + x7*:J J: + 1/2*x5*dJ, x5*J, c/3*x2*x5 + (-2*c + 2)/3*x0*x7], ('Gm', 'Gm'): []}
Bootstrap solution 1 of 3: free [x0, x2, x3], x5 = 0, x7 = 0
Bootstrap solution 2 of 3: free [x2, x5], x0 = c/3*x2^2, x3 = -x2, x7 = 0
Bootstrap solution 3 of 3: free [x5, x7], x0 = 0, x2 = 0, x3 = 0


## 6. $W(2,3,4)$

Three ansatz pairs, 73 unknowns, 27 triples (a few seconds). Fixing the two-point functions $c/3$, $c/4$ leaves the coupling $C_{33}^4$ (coefficient of $U$ in $W_{(1)}W$) determined up to sign.

In [10]:
E.<L, W, U> = NormallyOrderedExpressions(K)
V = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W],
                                  ('L', 'U'): [U.derivative(), 4*U]},
                           degrees={'L': 2, 'W': 3, 'U': 4})
B = OPEBootstrap(V)
NU = [x for x in B.unknowns() if B.describe(x) == 'coefficient of 1 in U_(7) U'][0]
B.fix({'x0': c/3, NU: c/4})
t = time.time(); B.impose_jacobi(); print('%.1f s' % (time.time() - t))
for S in B.solve():
    print(S)

0.7 s
Bootstrap solution 1 of 1: no free unknowns, x7^2 + (-112/15*c^2 - 2048/15*c - 1216/5)/(c^2 + 57/5*c + 154/5) = 0


## 7. Saving and resuming a bootstrap

Large bootstraps can take long, so they can be saved: `save(B, fn)` writes the algebra, the ansatz, the eliminated unknowns (with their values), the remaining equations and the imposed triples to a file, and `impose_jacobi(..., save=fn)` does so after every triple. `load(fn)` returns a bootstrap continuing from there, also in another session or on another machine (the file does not depend on the number of threads or processes); loading checks that the saved state belongs to the same algebra and ansatz.

First session: $W(2,3,4)$ without normalisations, imposing only the triples of total weight at most 10, saving after each one.

In [11]:
import tempfile
fn = os.path.join(tempfile.mkdtemp(), 'w234_bootstrap.sobj')   # any path

WTS = {'L': 2, 'W': 3, 'U': 4}

def w234():
    E.<L, W, U> = NormallyOrderedExpressions(K)
    return UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2], ('L', 'W'): [W.derivative(), 3*W],
                                         ('L', 'U'): [U.derivative(), 4*U]},
                                  degrees=WTS)

V = w234()
B = OPEBootstrap(V)
cheap = [(a, b, d) for a in WTS for b in WTS for d in WTS if WTS[a] + WTS[b] + WTS[d] <= 10]
B.impose_jacobi(cheap, save=fn)
print(B)

OPE bootstrap of Universal vertex algebra on generators ['L', 'W', 'U'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field: 73 unknowns in the OPEs of [('W', 'W'), ('W', 'U'), ('U', 'U')] (5 left), 4 equations, Jacobi identities imposed for 23 of 27 triples


`load(fn)` returns a new bootstrap (in a new session it also rebuilds `V` from the saved data). It knows which triples were imposed and continues with the others.

In [12]:
B2 = load(fn)
print(B2)
print(len(B2.imposed_triples()), 'triples imposed,', len(B2.unknowns()), 'unknowns left')
t = time.time(); B2.impose_jacobi(verbose=True); print('%.1f s' % (time.time() - t))
for S in B2.solve():
    print(S)

OPE bootstrap of Universal vertex algebra on generators ['L', 'W', 'U'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field: 73 unknowns in the OPEs of [('W', 'W'), ('W', 'U'), ('U', 'U')] (5 left), 4 equations, Jacobi identities imposed for 23 of 27 triples
23 triples imposed, 5 unknowns left


Jacobi (W, U, U): 46 failing identities, 3 unknowns left, 1 equations, 1.1 s


Jacobi (U, W, U): 46 failing identities, 3 unknowns left, 1 equations, 0.6 s


Jacobi (U, U, W): 37 failing identities, 3 unknowns left, 1 equations, 0.4 s


Jacobi (U, U, U): 50 failing identities, 3 unknowns left, 1 equations, 0.6 s


2.7 s
Bootstrap solution 1 of 2: free [x7, x55], x32 = 0
Bootstrap solution 2 of 2: free [x32, x55], x7 = 0


The same components as an uninterrupted run (the $W_4$ family with free `x7`, `x55`, and a degenerate one):

In [13]:
B3 = OPEBootstrap(w234())
B3.impose_jacobi()
print([str(S) for S in B3.solve()] == [str(S) for S in B2.solve()])

True


## References

- [Zam1985] A. B. Zamolodchikov, *Infinite additional symmetries in two-dimensional conformal quantum field theory*, Theoret. and Math. Phys. 65 (1985), 1205–1213, [doi:10.1007/BF01036128](https://doi.org/10.1007/BF01036128). — the $W_3$ algebra (sections 1–3).
- [KW1991] H. G. Kausch, G. M. T. Watts, *A study of W-algebras using Jacobi identities*, Nuclear Phys. B 354 (1991), 740–768, [doi:10.1016/0550-3213(91)90375-8](https://doi.org/10.1016/0550-3213(91)90375-8).
- [BFKNRV1991] R. Blumenhagen, M. Flohr, A. Kliem, W. Nahm, A. Recknagel, R. Varnhagen, *W-algebras with two and three generators*, Nuclear Phys. B 361 (1991), 255–289, [doi:10.1016/0550-3213(91)90624-7](https://doi.org/10.1016/0550-3213(91)90624-7). — bootstrapping W-algebras with Jacobi identities, with [KW1991]: $W(2,4)$, $W(2,5)$, $W(2,3,4)$ (sections 3, 4, 6).
- [Thi1991] K. Thielemans, *A Mathematica package for computing operator product expansions*, Internat. J. Modern Phys. C 2 (1991), 787–798, [doi:10.1142/S0129183191001001](https://doi.org/10.1142/S0129183191001001). — the OPE computations.